In [ ]:
import time
import csv
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
from typing import List, Dict, Optional, Tuple
from urllib.parse import quote, urljoin
import re
from datetime import datetime
import os


class PeopleNewsCrawler:
    def __init__(self, headless: bool = False):
        """
        初始化爬虫
        
        Args:
            headless: 是否使用无头模式
        """
        # 设置Chrome选项
        chrome_options = Options()
        if headless:
            chrome_options.add_argument("--headless")
        chrome_options.add_argument("--disable-gpu")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--window-size=1920,1080")
        chrome_options.add_argument("--disable-blink-features=AutomationControlled")
        chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
        chrome_options.add_experimental_option('useAutomationExtension', False)
        
        # 设置User-Agent
        user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
        chrome_options.add_argument(f'user-agent={user_agent}')
        
        # 初始化Chrome驱动
        self.driver = webdriver.Chrome(options=chrome_options)
        self.wait = WebDriverWait(self.driver, 10)
        
        # 设置时间限制
        self.year_limit = "2024"
        self.target_year = "2025"
        
        # 统计变量
        self.total_keywords = 0
        self.total_news = 0
        self.keyword_stats = {}
    
    def search_multiple_keywords(self, keyword_groups: Dict[str, List[str]]) -> Dict[str, List[Dict]]:
        """
        批量搜索多个关键词组
        
        Args:
            keyword_groups: 关键词组字典 {子议题: [关键词列表]}
            
        Returns:
            每个关键词的新闻数据字典 {关键词: 新闻列表}
        """
        all_results = {}
        self.total_keywords = sum(len(keywords) for keywords in keyword_groups.values())
        keyword_count = 0
        
        print(f"开始批量爬取 {self.total_keywords} 个关键词...")
        print("=" * 60)
        
        for subtopic, keywords in keyword_groups.items():
            print(f"\n📁 子议题: {subtopic}")
            print(f"🔑 关键词: {', '.join(keywords)}")
            print("-" * 40)
            
            for keyword in keywords:
                keyword_count += 1
                print(f"\n[{keyword_count}/{self.total_keywords}] 正在爬取关键词: '{keyword}'")
                
                try:
                    # 搜索当前关键词
                    news_data = self._search_single_keyword(keyword)
                    
                    if news_data:
                        all_results[keyword] = news_data
                        self.keyword_stats[keyword] = len(news_data)
                        self.total_news += len(news_data)
                        
                        print(f"✓ 关键词 '{keyword}' 找到 {len(news_data)} 条新闻")
                        
                        # 显示该关键词的前3条新闻
                        if len(news_data) > 0:
                            print(f"   最新3条:")
                            for i, news in enumerate(news_data[:3], 1):
                                print(f"     {i}. [{news.get('pubtime', '')}] {news.get('title', '')[:40]}...")
                    else:
                        print(f"✗ 关键词 '{keyword}' 没有找到新闻")
                        self.keyword_stats[keyword] = 0
                    
                    # 关键词之间等待一下，避免请求过快
                    time.sleep(2)
                    
                except Exception as e:
                    print(f"❌ 爬取关键词 '{keyword}' 时出错: {e}")
                    self.keyword_stats[keyword] = 0
        
        return all_results
    
    def _search_single_keyword(self, keyword: str) -> List[Dict]:
        """
        搜索单个关键词的新闻
        
        Args:
            keyword: 搜索关键词
            
        Returns:
            新闻数据列表
        """
        all_news = []
        reached_2024 = False
        consecutive_empty_pages = 0
        max_consecutive_empty = 3
        
        try:
            # 构建搜索URL
            encoded_keyword = quote(keyword.encode('utf-8'))
            search_url = f"http://search.people.cn/s?keyword={encoded_keyword}&st=0"
            
            print(f"   访问: {search_url}")
            self.driver.get(search_url)
            
            # 等待页面加载
            time.sleep(3)
            
            # 点击"精确匹配"复选框
            self._click_exact_match()
            
            # 等待搜索结果重新加载
            time.sleep(2)
            
            # 开始自动翻页抓取
            current_page = 1
            
            while not reached_2024:
                # 获取当前页面的HTML
                page_source = self.driver.page_source
                
                # 解析当前页面的新闻
                news_items, reached_2024 = self._parse_current_page(page_source, keyword)
                
                if news_items:
                    all_news.extend(news_items)
                    consecutive_empty_pages = 0
                    print(f"   第 {current_page} 页: {len(news_items)} 条")
                else:
                    consecutive_empty_pages += 1
                    if consecutive_empty_pages >= max_consecutive_empty:
                        print(f"   连续{max_consecutive_empty}页无数据，停止")
                        break
                
                # 检查是否到达2024年
                if reached_2024:
                    print(f"   遇到{self.year_limit}年新闻，停止")
                    break
                
                # 尝试点击下一页
                if self._go_to_next_page():
                    current_page += 1
                    time.sleep(2)
                else:
                    print(f"   已到达最后一页，共 {current_page} 页")
                    break
            
            # 按时间排序（从新到旧）
            all_news.sort(key=lambda x: x.get('pubtime', ''), reverse=True)
            
        except Exception as e:
            print(f"   搜索过程中出错: {e}")
        
        return all_news
    
    def _click_exact_match(self):
        """
        点击"精确匹配"复选框
        """
        try:
            # 根据提供的HTML结构查找精确匹配复选框
            exact_match_xpath = "//span[@class='el-checkbox_label' and contains(text(), '精确匹配')]"
            exact_match_element = self.wait.until(
                EC.presence_of_element_located((By.XPATH, exact_match_xpath))
            )
            
            # 找到父级label元素并点击
            label_element = exact_match_element.find_element(By.XPATH, "./parent::label")
            
            if label_element.is_displayed() and label_element.is_enabled():
                # 使用JavaScript点击，更可靠
                self.driver.execute_script("arguments[0].click();", label_element)
                time.sleep(1)
                
        except TimeoutException:
            # 尝试其他选择器
            try:
                checkbox = self.driver.find_element(By.CSS_SELECTOR, "input[type='checkbox']")
                self.driver.execute_script("arguments[0].click();", checkbox)
            except:
                pass
    
    def _parse_current_page(self, html: str, keyword: str) -> tuple:
        """
        解析当前页面的新闻
        
        Args:
            html: 页面HTML
            keyword: 搜索关键词
            
        Returns:
            (新闻数据列表, 是否到达2024年)
        """
        soup = BeautifulSoup(html, 'html.parser')
        news_list = []
        reached_2024 = False
        
        # 根据提供的HTML结构查找新闻列表
        article_list = soup.find('ul', class_='article')
        if not article_list:
            article_list = soup.find('div', class_='list-wrapper')
        
        if article_list:
            news_items = article_list.find_all('li', class_='clear')
            for item in news_items:
                try:
                    news_data, should_stop = self._parse_news_item(item, keyword)
                    if news_data:
                        news_list.append(news_data)
                        if should_stop:
                            reached_2024 = True
                except:
                    continue
        
        return news_list, reached_2024
    
    def _parse_news_item(self, item, keyword: str) -> tuple:
        """
        解析单个新闻项
        
        Args:
            item: 新闻项的HTML元素
            keyword: 搜索关键词
            
        Returns:
            (新闻数据字典, 是否应该停止)
        """
        try:
            # 查找包含标题的div
            content_div = item.find('div', class_='content')
            if not content_div:
                return None, False
            
            # 提取标题
            title_div = content_div.find('div', class_='ttl')
            if not title_div:
                title_div = content_div.find('div', class_='tt1')
            
            if title_div:
                title_elem = title_div.find('a', href=True)
                if title_elem:
                    title = title_elem.get_text(strip=True)
                    link = title_elem['href']
                else:
                    return None, False
            else:
                title_elem = content_div.find('a', href=True)
                if not title_elem:
                    return None, False
                title = title_elem.get_text(strip=True)
                link = title_elem['href']
            
            # 确保链接是完整的URL
            if not link.startswith('http'):
                link = urljoin('http://www.people.com.cn', link)
            
            # 提取摘要
            abstract_elem = content_div.find('div', class_='abs')
            if abstract_elem:
                for em in abstract_elem.find_all('em'):
                    em.replace_with(em.get_text())
                abstract = abstract_elem.get_text(strip=True)
            else:
                abstract = ""
            
            # 提取发布时间和来源
            pubtime = ""
            source = ""
            should_stop = False
            
            fot_elem = content_div.find('div', class_='fot')
            if fot_elem:
                # 提取时间
                time_elem = fot_elem.find('span', class_='tip-pubtime')
                if time_elem:
                    pubtime = time_elem.get_text(strip=True)
                
                if not pubtime:
                    time_pattern = r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}'
                    fot_text = fot_elem.get_text()
                    time_match = re.search(time_pattern, fot_text)
                    if time_match:
                        pubtime = time_match.group()
                
                # 检查时间是否达到限制年份
                if pubtime and pubtime.startswith(self.year_limit):
                    should_stop = True
                
                # 提取来源
                source_elem = fot_elem.find('a', class_='tip-source')
                if source_elem:
                    source_text = source_elem.get_text(strip=True)
                    source = self._clean_source_text(source_text)
                else:
                    fot_text = fot_elem.get_text(strip=True)
                    if pubtime:
                        fot_text = fot_text.replace(pubtime, "").strip()
                    fot_text = fot_text.replace("来源：", "").strip()
                    source = self._clean_source_text(fot_text)
            
            # 检查是否包含关键词
            if keyword in title or keyword in abstract:
                if not source or len(source) < 2 or source in ["蒙", "来", "源"]:
                    source = "未知栏目"
                
                # 只保留2025年的新闻
                if pubtime and pubtime.startswith(self.target_year):
                    news_data = {
                        'keyword': keyword,
                        'pubtime': pubtime,
                        'title': title,
                        'abstract': abstract,
                        'source': source,
                        'link': link
                    }
                    return news_data, should_stop
                else:
                    return None, should_stop
            
            return None, False
            
        except Exception as e:
            return None, False
    
    def _clean_source_text(self, source_text: str) -> str:
        """
        清理来源文本
        """
        if not source_text:
            return ""
        
        source_text = source_text.strip()
        source_text = re.sub(r'^来源[:：]\s*', '', source_text)
        source_text = re.sub(r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}', '', source_text)
        source_text = re.sub(r'\s+', ' ', source_text).strip()
        
        if len(source_text) < 2:
            return ""
        
        if source_text in ["蒙", "来", "源", "地", "方", "新", "闻"]:
            return ""
        
        return source_text
    
    def _go_to_next_page(self) -> bool:
        """
        点击下一页按钮
        """
        try:
            time.sleep(1)
            
            # 方法1：查找class="page-next"的span
            next_page_spans = self.driver.find_elements(By.CSS_SELECTOR, "span.page-next")
            
            if next_page_spans:
                for span in next_page_spans:
                    if span.is_displayed():
                        self.driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", span)
                        time.sleep(0.5)
                        self.driver.execute_script("arguments[0].click();", span)
                        return True
            
            # 方法2：查找包含"下一页"文本的元素
            next_buttons = self.driver.find_elements(By.XPATH, 
                "//*[contains(text(), '下一页') or contains(text(), '下一頁')]")
            
            for button in next_buttons:
                if button.is_displayed():
                    self.driver.execute_script("arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", button)
                    time.sleep(0.5)
                    self.driver.execute_script("arguments[0].click();", button)
                    return True
            
            return False
            
        except Exception as e:
            return False
    
    def save_to_csv(self, all_results: Dict[str, List[Dict]], output_dir: str = "output"):
        """
        保存所有关键词的新闻数据到CSV文件
        
        Args:
            all_results: 所有关键词的新闻数据字典
            output_dir: 输出目录
        """
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        
        timestamp = time.strftime("%Y%m%d_%H%M%S")
        
        # 1. 保存每个关键词的单独CSV文件
        for keyword, news_data in all_results.items():
            if news_data:
                # 清理文件名中的特殊字符
                safe_keyword = re.sub(r'[<>:"/\\|?*]', '_', keyword)
                filename = os.path.join(output_dir, f"人民网_{safe_keyword}_{self.target_year}年_{len(news_data)}条_{timestamp}.csv")
                
                fieldnames = ['keyword', 'pubtime', 'title', 'abstract', 'source', 'link']
                
                with open(filename, 'w', encoding='utf-8-sig', newline='') as f:
                    writer = csv.DictWriter(f, fieldnames=fieldnames)
                    writer.writeheader()
                    
                    for item in news_data:
                        row = {field: item.get(field, '') for field in fieldnames}
                        writer.writerow(row)
        
        # 2. 保存合并的总CSV文件（所有关键词）
        all_news = []
        for news_data in all_results.values():
            all_news.extend(news_data)
        
        if all_news:
            # 按时间排序
            all_news.sort(key=lambda x: x.get('pubtime', ''), reverse=True)
            
            # 总文件
            total_filename = os.path.join(output_dir, f"人民网_所有关键词_{self.target_year}年_{self.total_news}条_{timestamp}.csv")
            
            fieldnames = ['keyword', 'pubtime', 'title', 'abstract', 'source', 'link']
            
            with open(total_filename, 'w', encoding='utf-8-sig', newline='') as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
                
                for item in all_news:
                    row = {field: item.get(field, '') for field in fieldnames}
                    writer.writerow(row)
            
            print(f"\n✓ 合并总文件已保存: {total_filename}")
        
        # 3. 保存统计报告
        stats_filename = os.path.join(output_dir, f"统计报告_{timestamp}.txt")
        with open(stats_filename, 'w', encoding='utf-8') as f:
            f.write("人民网新闻爬取统计报告\n")
            f.write("=" * 60 + "\n")
            f.write(f"爬取时间: {time.strftime('%Y-%m-%d %H:%M:%S')}\n")
            f.write(f"目标年份: {self.target_year}\n")
            f.write(f"关键词总数: {self.total_keywords}\n")
            f.write(f"新闻总数: {self.total_news}\n")
            f.write("=" * 60 + "\n\n")
            
            # 按子议题分组统计
            f.write("📊 按子议题统计:\n")
            f.write("-" * 60 + "\n")
            
            # 这里需要子议题信息，暂时先按关键词统计
            f.write("📊 按关键词统计:\n")
            f.write("-" * 60 + "\n")
            
            # 按数量排序
            sorted_stats = sorted(self.keyword_stats.items(), key=lambda x: x[1], reverse=True)
            for keyword, count in sorted_stats:
                f.write(f"{keyword:20} : {count:4} 条\n")
        
        print(f"✓ 统计报告已保存: {stats_filename}")
        
        # 显示统计信息
        self._show_statistics(all_results)
    
    def _show_statistics(self, all_results: Dict[str, List[Dict]]):
        """
        显示统计信息
        """
        print("\n" + "=" * 60)
        print("📊 爬取统计报告")
        print("=" * 60)
        print(f"爬取时间: {time.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"目标年份: {self.target_year}")
        print(f"关键词总数: {self.total_keywords}")
        print(f"找到新闻的关键词: {sum(1 for v in all_results.values() if v)}")
        print(f"新闻总数: {self.total_news}")
        print("-" * 60)
        
        # 按数量排序显示
        print("关键词排名 (按新闻数量):")
        print("-" * 60)
        
        sorted_stats = sorted(self.keyword_stats.items(), key=lambda x: x[1], reverse=True)
        for i, (keyword, count) in enumerate(sorted_stats, 1):
            if count > 0:
                bar = "█" * (count // 10 + 1)  # 每10条一个█
                print(f"{i:2}. {keyword:15} : {count:4} 条 {bar}")
        
        print("=" * 60)
    
    def close(self):
        """关闭浏览器驱动"""
        if self.driver:
            self.driver.quit()
            print("✅ 浏览器已关闭")


def get_keyword_groups() -> Dict[str, List[str]]:
    """
    从文档中提取关键词组
    
    Returns:
        关键词组字典 {子议题: [关键词列表]}
    """
    keyword_groups = {
        "加班文化与劳动制度": ["加班", "996", "劳动法", "加班费"],
        "职场压力与焦虑": ["工作压力", "职场焦虑", "职场压力", "工作焦虑"],
        "职业倦怠与过劳": ["工作倦怠", "过劳肥", "工作疲惫", "职业疲惫", "工作躺平"],
        "企业责任与员工关怀": ["企业责任", "员工关怀", "职业健康"],
        "职场文化与内卷现象": ["职场内卷", "职场竞争", "职场PUA", "职场霸凌"],
        "职场心理健康支持": ["职场心理咨询", "职场心理健康", "职场关怀"]
    }
    
    return keyword_groups


def main():
    """主函数"""
    crawler = None
    
    try:
        # 获取关键词组
        keyword_groups = get_keyword_groups()
        
        # 创建爬虫实例
        crawler = PeopleNewsCrawler(headless=False)
        
        print("=" * 60)
        print("人民网新闻爬虫 - 批量关键词爬取")
        print("=" * 60)
        
        # 显示关键词组信息
        print("📋 关键词组配置:")
        print("-" * 60)
        for subtopic, keywords in keyword_groups.items():
            print(f"📁 {subtopic}:")
            print(f"   {', '.join(keywords)}")
        print("=" * 60)
        
        # 开始批量爬取
        start_time = time.time()
        
        all_results = crawler.search_multiple_keywords(keyword_groups)
        
        end_time = time.time()
        elapsed_time = end_time - start_time
        
        if any(all_results.values()):
            # 创建输出目录
            timestamp = time.strftime("%Y%m%d_%H%M%S")
            output_dir = f"人民网新闻_{crawler.target_year}年_{timestamp}"
            
            # 保存所有数据
            print("\n" + "=" * 60)
            print("💾 正在保存数据...")
            print("=" * 60)
            
            crawler.save_to_csv(all_results, output_dir)
            
            print("\n" + "=" * 60)
            print("✅ 批量爬取完成！")
            print(f"⏱️  总用时: {elapsed_time:.1f} 秒")
            print(f"📈 平均速度: {crawler.total_news / elapsed_time:.1f} 条/秒" if elapsed_time > 0 else "")
            print(f"📂 数据保存到目录: {output_dir}")
            print("=" * 60)
            
        else:
            print("\n❌ 没有找到任何新闻！")
            
    except KeyboardInterrupt:
        print("\n\n⚠️ 用户中断操作")
        
    except Exception as e:
        print(f"\n❌ 程序执行出错: {e}")
        import traceback
        traceback.print_exc()
        
    finally:
        # 确保关闭浏览器
        if crawler:
            crawler.close()


if __name__ == "__main__":
    print("🔧 人民网新闻爬虫 - 批量关键词爬取")
    print("=" * 60)
    print("📋 功能特点:")
    print("  1. 批量爬取多个关键词")
    print("  2. 每个关键词生成单独CSV文件")
    print("  3. 生成合并总文件和统计报告")
    print("  4. 只抓取2025年新闻，遇到2024年停止")
    print("=" * 60)
    
    # 询问是否开始
    input("\n🎯 按Enter键开始批量爬取，或Ctrl+C退出...")
    
    main()